# Correlated Wishart Ensemble (CWE) — Constant Correlation
**Author:** Elsa Susana Ochoa  
**Master's Thesis — UNAM, 2018**

Generates the Correlated Wishart Ensemble using a constant background correlation estimated from the empirical S&P 500 correlation matrices. Applies Power Mapping and computes eigenvalue/eigenvector statistics.

## 1. Imports

In [ ]:
import numpy as np
from numpy import linalg as LA
import matplotlib.pyplot as plt
import scipy
from scipy import linalg
from scipy import stats
from scipy.optimize import curve_fit
from pylab import *
%matplotlib inline

## 2. Load Data
Load symbols, dates, GICS sectors and prices for 293 S&P 500 stocks (1992-2014).

In [ ]:
with open('../data/Symbols.txt', 'r') as S:
    s = [line.split()[0] for line in S]
lensymbols = len(s)

with open('../data/Dates.txt', 'r') as D:
    d = [line.split()[0] for line in D]
lendates = len(d)

with open('../data/Gics.txt', 'r') as G:
    g = [line.split()[0] for line in G]

Prices = np.loadtxt('../data/Prices.txt')
lenPrices = len(Prices)

print(f'Stocks: {lensymbols}, Dates: {lendates}, Price matrix shape: {Prices.shape}')

## 3. Compute Returns
Daily returns: $R_t = (P_{t} / P_{t-1}) - 1$

In [ ]:
Returns = []
for i in range(lensymbols):
    Returns.append([])
    for j in range(1, lendates):
        r = (Prices[i][j] / Prices[i][j-1]) - 1
        Returns[-1].append(r)
Returns = np.asarray(Returns)
print(f'Returns shape: {Returns.shape}')

## 4. Parameters
Define parameters for the CWE simulation.

In [ ]:
nummat    = 2000   # Number of random matrices per window
N         = 293    # Number of stocks
T         = 44     # Window size in days
e         = 0.05   # Power Mapping exponent offset
q         = 1 + e  # Power Mapping exponent (1 < q < 2)
numVentana = 262   # Number of windows

C         = np.empty((N, N))
Cq        = np.empty((N, N))
Corr      = np.empty((N, N))
Corrsqr   = np.empty((N, N))
CWOEvalores = np.empty((numVentana, N))
v1total   = np.empty((nummat, N, N))
w1total   = np.empty((nummat, N))

print(f'q = {q:.3f}, nummat = {nummat}, N = {N}, T = {T}')

## 5. Compute Mean Correlation per Window
Estimate the background correlation from the empirical correlation matrices.
This is used as the constant correlation C_0 in the CWE.

In [ ]:
Cmean = []
for i in range(numVentana):
    C = np.loadtxt('../data/MatrizCorr%d.dat' % i)
    cm = (C.sum() - N) / (N**2 - N)  # Mean off-diagonal correlation
    Cmean.append(cm)

plt.plot(Cmean)
plt.xlabel('Window')
plt.ylabel('Mean correlation')
plt.title('Background correlation per window')
plt.tight_layout()
plt.savefig('../figures/mean_correlation.png', bbox_inches='tight')
plt.show()
print(f'Mean correlation range: [{min(Cmean):.3f}, {max(Cmean):.3f}]')

## 6. Correlated Wishart Ensemble — Constant Correlation
For each window:
1. Build background correlation matrix C_0 with constant off-diagonal correlation
2. Generate nummat random matrices: $W = C_0^{(1/2)} X X^T C_0^{(1/2)} / T$
3. Apply Power Mapping: $sign(rho)|rho|^q$
4. Diagonalize and store eigenvalues

In [1]:
for ii in range(numVentana):
    corr1 = Cmean[ii]  # Background correlation for this window
    Wmean = []

    # Build constant correlation matrix C_0
    for i in range(N):
        for j in range(i, N):
            Corr[i][j] = 1.0 if i == j else corr1
            Corr[j][i] = Corr[i][j]

    # Matrix square root of C_0
    Corrsqr = np.real(scipy.linalg.sqrtm(Corr))

    for matriz in range(nummat):
        # Generate random matrix
        M1 = np.random.randn(N, T)

        # Apply background correlation: W = C_0^(1/2) M
        CorrsqrM1 = np.dot(Corrsqr, M1)

        # Normalize rows
        CorrsqrM1 = np.array([(s - s.mean()) / s.std() for s in CorrsqrM1])

        # Wishart matrix
        C = np.dot(CorrsqrM1, CorrsqrM1.T) / T

        # Power Mapping: sign(rho)|rho|^q
        Cq = np.sign(C) * (np.abs(C) ** q)

        # Diagonalize
        w1, v1 = LA.eigh(Cq)
        w1total[matriz] = w1
        v1total[matriz] = v1

    # Average eigenvalues across all random matrices
    for jj in range(N):
        Wmean.append(w1total[:, jj].mean())
    CWOEvalores[ii] = np.asarray(Wmean)

    if ii % 50 == 0:
        print(f'Window {ii}/{numVentana} done')

print('CWE computation complete.')

NameError: name 'numVentana' is not defined

## 7. Save Results

In [ ]:
import os
os.makedirs('../data', exist_ok=True)

np.save('../data/CWOEcte_q%.2f' % q, CWOEvalores)
print(f'Saved CWOEvalores shape: {CWOEvalores.shape}')

## 8. Visualization
Plot eigenvalue spectrum and Participation Ratio of the CWE.

In [ ]:
import os
os.makedirs('../figures', exist_ok=True)

# Eigenvalue spectrum for a single window
plt.hist(CWOEvalores[0][:N-T+1], 40, density=True, color='steelblue')
plt.xlabel('Eigenvalue')
plt.ylabel('Density')
plt.title(f'CWE Eigenvalue Spectrum (window 0, q={q:.2f})')
plt.tight_layout()
plt.savefig('../figures/CWE_eigenvalue_spectrum.png', bbox_inches='tight')
plt.show()

## 9. Participation Ratio (PR)
Measures how many stocks contribute to each eigenvector mode.

PR_k = 1 / (N * sum_i u_ki^4)

- PR near 1: all stocks contribute equally (market mode)
- PR near 0: few stocks dominate (sector or individual mode)

In [ ]:
# Participation Ratio for CWE eigenvectors
vIPR = np.sum(v1total**4, axis=1)
vIPR = 1 / (N * vIPR)

plt.plot(vIPR.mean(axis=0))
plt.xlabel('Eigenvector index')
plt.ylabel('Participation Ratio')
plt.title(f'CWE Participation Ratio (q={q:.2f})')
plt.tight_layout()
plt.savefig('../figures/CWE_participation_ratio.png', bbox_inches='tight')
plt.show()